<a href="https://colab.research.google.com/github/aiman0642/saas-retention-intelligence/blob/main/04_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 — Data Preprocessing

**Project:** SaaS Customer Retention Intelligence System

**Goal of this notebook:** turn the account-level analysis table from
Module 2 into a clean, modeling-ready dataset — handle missing values,
check for data leakage, encode categoricals, scale numerics, and produce
a train/test split. This is preparation for Module 6 (Machine Learning),
not modeling itself.



## 4.1 Mount Google Drive & Load Data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

PROJECT_DIR = "/content/drive/MyDrive/saas-retention-intelligence"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

df = pd.read_csv(f"{PROCESSED_DIR}/account_view_eda.csv", parse_dates=["signup_date", "churn_date"])
print(f"Loaded account_view_eda: {df.shape}")
df.head(3)

Loaded account_view_eda: (500, 30)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,plan_tier_sub,mrr_amount,arr_amount,billing_frequency,auto_renew_flag,upgrade_flag,downgrade_flag,total_usage_count,total_usage_duration_secs,total_errors,distinct_features_used,ticket_count,avg_satisfaction,avg_resolution_hours,escalation_count,churn_date,tenure_days,tenure_bucket,usage_bucket,mrr_bucket
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,Basic,836,10032,monthly,True,False,False,535,152339,38,50,2.0,3.000000,23.000000,0.0,2024-11-23,38,0-3mo,Medium-High,Medium-Low
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,Pro,882,10584,monthly,True,False,False,355,101136,14,32,3.0,4.000000,38.000000,0.0,NaT,502,1-2yr,Low,Medium-Low
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,Pro,98,1176,annual,True,False,False,821,251210,48,78,3.0,4.666667,43.666667,0.0,2024-10-06,40,0-3mo,High,Low


## 4.2 Recap Columns & Dtypes

Quick refresher before making any changes — what's actually in this
table, and which columns are numerical vs categorical vs identifiers.

In [3]:
print(df.dtypes)
print()
print(f"Target variable: churn_flag  |  positive rate: {df['churn_flag'].mean():.1%}")

account_id                           object
account_name                         object
industry                             object
country                              object
signup_date                  datetime64[ns]
referral_source                      object
plan_tier                            object
seats                                 int64
is_trial                               bool
churn_flag                             bool
plan_tier_sub                        object
mrr_amount                            int64
arr_amount                            int64
billing_frequency                    object
auto_renew_flag                        bool
upgrade_flag                           bool
downgrade_flag                         bool
total_usage_count                     int64
total_usage_duration_secs             int64
total_errors                          int64
distinct_features_used                int64
ticket_count                        float64
avg_satisfaction                

## 4.3 Handle Missing Values

`avg_satisfaction` and `avg_resolution_hours` are NaN for accounts with
zero support tickets — that's a real "no tickets filed" state, not a data
quality gap. Fill with a sentinel that a model can distinguish from a
real low/high value, and add a companion "has_tickets" flag so the
distinction isn't lost.

In [4]:
missing_before = df.isnull().sum()
print("Missing values before handling:")
print(missing_before[missing_before > 0])

Missing values before handling:
avg_satisfaction         34
avg_resolution_hours      8
churn_date              148
tenure_bucket             3
dtype: int64


In [5]:
df["has_support_tickets"] = (df["ticket_count"] > 0).astype(int)

# Median-impute satisfaction/resolution time among accounts that DID have
# tickets, rather than using 0 (which would misleadingly look like a very
# low satisfaction score).
df["avg_satisfaction"] = df["avg_satisfaction"].fillna(df["avg_satisfaction"].median())
df["avg_resolution_hours"] = df["avg_resolution_hours"].fillna(df["avg_resolution_hours"].median())

print("\nMissing values after handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Missing values after handling:
churn_date       148
tenure_bucket      3
dtype: int64


## 4.4 Data Leakage Check

This is the most important step in this notebook. Anything that would
only be known *after* an account churns cannot be used to predict churn.

- `churn_date` — directly reveals the target (an account has a date only
  if it churned). **Drop.**
- `tenure_days` — for churned accounts this was computed using
  `churn_date` (Module 2, section 2.2). This is a real, known modeling
  tradeoff: tenure-at-churn is legitimate as a *retrospective* feature
  for understanding churn patterns, but it's borderline leakage for a
  model meant to score *currently active* accounts, since an active
  account's tenure keeps growing and isn't "final" the way a churned
  account's is. **Decision: keep `tenure_days` for this project since the
  goal is descriptive/explanatory (SHAP-driven), not a live scoring
  pipeline — but this tradeoff is called out explicitly here and again in
  the README's Limitations section, not hidden.**
- `tenure_bucket`, `usage_bucket`, `mrr_bucket` — these are just binned
  versions of features already in the table (`tenure_days`,
  `total_usage_count`, `mrr_amount`). Keeping both the raw and binned
  version is redundant, not leakage — drop the bucket columns and let
  the model work from the raw numeric values instead.
- `signup_month` (if present) — not present in this table; no action.

In [6]:
leakage_cols = ["churn_date"]
redundant_cols = ["tenure_bucket", "usage_bucket", "mrr_bucket"]

df = df.drop(columns=[c for c in leakage_cols + redundant_cols if c in df.columns])
print(f"Dropped leakage columns: {leakage_cols}")
print(f"Dropped redundant binned columns: {redundant_cols}")
print(f"Shape after drop: {df.shape}")

Dropped leakage columns: ['churn_date']
Dropped redundant binned columns: ['tenure_bucket', 'usage_bucket', 'mrr_bucket']
Shape after drop: (500, 27)


## 4.5 Remove Identifier & Redundant Duplicate Columns

IDs and free-text names carry no generalizable signal for a model — keep
them out of the feature set (though `account_id` is retained separately
so predictions can be joined back to specific accounts later).

`plan_tier_sub` is also dropped here: Module 2's merge between `accounts`
and `subscriptions` produced two `plan_tier` columns (account-level and
subscription-level), and pandas auto-suffixed the second one
`plan_tier_sub` rather than erroring. It duplicates the same information
already captured in `plan_tier` — keeping both would double-count the
same signal and, since it's a raw string column, would break sklearn
models in Module 6 if left unencoded.

In [7]:
id_cols = ["account_id", "account_name"]
duplicate_cols = ["plan_tier_sub"]
account_ids = df["account_id"].copy()  # keep for later re-joining, e.g. in the dashboard

df_model = df.drop(columns=[c for c in id_cols + duplicate_cols if c in df.columns])
df_model = df_model.drop(columns=["signup_date"], errors="ignore")  # raw date not usable directly by most models

print(f"Modeling table shape: {df_model.shape}")
print(f"Columns: {list(df_model.columns)}")

Modeling table shape: (500, 23)
Columns: ['industry', 'country', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag', 'mrr_amount', 'arr_amount', 'billing_frequency', 'auto_renew_flag', 'upgrade_flag', 'downgrade_flag', 'total_usage_count', 'total_usage_duration_secs', 'total_errors', 'distinct_features_used', 'ticket_count', 'avg_satisfaction', 'avg_resolution_hours', 'escalation_count', 'tenure_days', 'has_support_tickets']


## 4.6 Encode Categorical Variables

One-hot encode nominal categoricals (no inherent order): `industry`,
`country`, `plan_tier`, `billing_frequency`, `referral_source`. Boolean
columns (`is_trial`, `auto_renew_flag`, etc.) are already 0/1-compatible
and don't need encoding.

In [8]:
categorical_cols = ["industry", "country", "plan_tier", "billing_frequency", "referral_source"]
categorical_cols = [c for c in categorical_cols if c in df_model.columns]

print(f"One-hot encoding: {categorical_cols}")
for col in categorical_cols:
    print(f"  {col}: {df_model[col].nunique()} categories")

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
print(f"\nShape after encoding: {df_model.shape}")

One-hot encoding: ['industry', 'country', 'plan_tier', 'billing_frequency', 'referral_source']
  industry: 5 categories
  country: 7 categories
  plan_tier: 3 categories
  billing_frequency: 2 categories
  referral_source: 5 categories

Shape after encoding: (500, 35)


## 4.7 Confirm Boolean Columns Are Numeric

`pd.get_dummies` and most sklearn models expect 0/1, not Python `True`/
`False` objects — cast explicitly so nothing downstream breaks silently.

In [9]:
bool_cols = df_model.select_dtypes(include="bool").columns.tolist()
df_model[bool_cols] = df_model[bool_cols].astype(int)
print(f"Cast to int: {bool_cols}")

Cast to int: ['is_trial', 'churn_flag', 'auto_renew_flag', 'upgrade_flag', 'downgrade_flag', 'industry_DevTools', 'industry_EdTech', 'industry_FinTech', 'industry_HealthTech', 'country_CA', 'country_DE', 'country_FR', 'country_IN', 'country_UK', 'country_US', 'plan_tier_Enterprise', 'plan_tier_Pro', 'billing_frequency_monthly', 'referral_source_event', 'referral_source_organic', 'referral_source_other', 'referral_source_partner']


## 4.8 Train / Test Split

Stratify on `churn_flag` so both splits preserve the same ~22% churn
rate — important with an imbalanced target, otherwise a random split
could accidentally concentrate churners in one side.

In [10]:
X = df_model.drop(columns=["churn_flag"])
y = df_model["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}   y_train churn rate: {y_train.mean():.1%}")
print(f"X_test:  {X_test.shape}   y_test churn rate: {y_test.mean():.1%}")

X_train: (400, 34)   y_train churn rate: 22.0%
X_test:  (100, 34)   y_test churn rate: 22.0%


## 4.9 Scale Numerical Features

Logistic Regression (Module 6's baseline model) is sensitive to feature
scale; tree-based models (Decision Tree, Random Forest) are not, but
scaling doesn't hurt them either. Fit the scaler on the **training set
only** and apply it to test — fitting on the full dataset would leak
test-set distribution info into training.

In [11]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
# Don't scale already-binary 0/1 columns (encoded categoricals, boolean flags)
numeric_cols = [c for c in numeric_cols if X_train[c].nunique() > 2]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"Scaled columns ({len(numeric_cols)}): {numeric_cols}")

Scaled columns (12): ['seats', 'mrr_amount', 'arr_amount', 'total_usage_count', 'total_usage_duration_secs', 'total_errors', 'distinct_features_used', 'ticket_count', 'avg_satisfaction', 'avg_resolution_hours', 'escalation_count', 'tenure_days']


## 4.10 Final Dtype Safety Check

Confirm every column feeding the model is actually numeric before
saving — catches any leftover string/object columns (like the
`plan_tier_sub` issue above) before they reach Module 6 and cause a
harder-to-diagnose error there instead of here.

In [12]:
non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric columns remain in X_train, fix before proceeding: {non_numeric}")
print("All columns in X_train are numeric — safe to proceed.")

All columns in X_train are numeric — safe to proceed.


## 4.11 Save Preprocessed Datasets

Save both the unscaled (for tree-based models, which don't need scaling
and are easier to interpret in original units) and scaled (for Logistic
Regression) versions, plus the fitted scaler for reuse in Module 9
(SHAP) and the Streamlit app.

In [14]:
import joblib

X_train.to_csv(f"{PROCESSED_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{PROCESSED_DIR}/X_test.csv", index=False)
X_train_scaled.to_csv(f"{PROCESSED_DIR}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{PROCESSED_DIR}/X_test_scaled.csv", index=False)
y_train.to_csv(f"{PROCESSED_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{PROCESSED_DIR}/y_test.csv", index=False)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
joblib.dump(scaler, f"{PROJECT_DIR}/models/scaler.pkl")

print("Saved: X_train.csv, X_test.csv, X_train_scaled.csv, X_test_scaled.csv, y_train.csv, y_test.csv")
print(f"Saved fitted scaler to {PROJECT_DIR}/models/scaler.pkl")

Saved: X_train.csv, X_test.csv, X_train_scaled.csv, X_test_scaled.csv, y_train.csv, y_test.csv
Saved fitted scaler to /content/drive/MyDrive/saas-retention-intelligence/models/scaler.pkl


## 4.12 Module 4 Summary

- Loaded the Module 2 account-level table and confirmed missing-value
  handling (support ticket fields — genuine "no tickets" cases, not gaps)
- **Leakage check:** dropped `churn_date` outright; kept `tenure_days`
  as a deliberate, documented tradeoff for this descriptive/explanatory
  project rather than a live-scoring pipeline
- Dropped identifier columns and a redundant duplicate (`plan_tier_sub`,
  a leftover from Module 2's merge) that would have broken sklearn models
  in Module 6 if left unencoded
- One-hot encoded categoricals, cast booleans to int
- Stratified 80/20 train/test split preserving the ~22% churn rate
- Scaled numeric features (fit on train only) and saved both scaled and
  unscaled versions for different model types
- Verified with an explicit dtype check that every modeling column is
  numeric before saving

**Next:** Module 5 — Feature Engineering